# Add your own rule

A *rule* is a Python function that looks at a candidate and says pass/fail with a measured value and a sentence. Register it, list the file as a plugin, name it in the family's `constraints` — the search enforces it and every report explains it.

In [ ]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu
!pip -q install git+https://github.com/ABnano/MEIDNet.git


In [ ]:
%%writefile my_rules.py
from meidnet.constraints import CONSTRAINTS

@CONSTRAINTS.register('no_lead', 'Rejects any composition containing Pb.')
def no_lead(cand):
    return cand.result('no_lead', 'Pb' not in cand.elements.values())

@CONSTRAINTS.register('mass_window', 'Mean atomic mass must be between min and max (g/mol).')
def mass_window(cand, min=0.0, max=200.0):
    from pymatgen.core import Element
    m = sum(Element(e).atomic_mass for e in cand.structure.species) / len(cand.structure)
    return cand.result('mass_window', min <= m <= max, float(m), (min, max), f'mean mass {m:.1f}')


## Use the rules in a family file
Copy the built-in family and append the rules.

In [ ]:
import meidnet.family, shutil, os
src = os.path.join(os.path.dirname(meidnet.family.__file__), 'families', 'perovskite_abx3.yaml')
text = open(src).read().replace('name: perovskite_abx3', 'name: my_perovskite')
text += '  - {name: no_lead}\n  - {name: mass_window, min: 0, max: 120}\n'
open('my_perovskite.yaml', 'w').write(text)
print(text[-600:])

In [ ]:
%%writefile meidnet.yaml
name: rules_demo
model_path: /root/.meidnet/dual_autoencoder_clip_earlyfusion_propertyaware_2k.pth
plugins: [my_rules.py]
generation:
  family: my_perovskite.yaml
  variant: halide
  objectives:
    - {property: dir_gap, loss: l2, weight: 10000}
    - {property: heat_all, loss: at_most, weight: 6000, select_weight: 0.4}
  targets: [{dir_gap: 1.8, heat_all: -0.1}]
  per_target: 3
  population: 24
  rounds: 4
  steps: 200


In [ ]:
!meidnet demo --no-open -n 1 --rounds 1 --steps 20 --device cpu > /dev/null  # downloads the checkpoint
!meidnet generate meidnet.yaml

In [ ]:
from IPython.display import HTML, display
display(HTML(open('runs/rules_demo/generation_report.html', encoding='utf-8').read()))

Every candidate card now lists **No lead** and **mass window** with the measured value — the same function decided acceptance and wrote the explanation.